## 00_utils — RCM Intelligence Platform
### Purpose
Shared helper functions used across every notebook in the pipeline.
Import this after config in every notebook:
```
%run ../00_setup/00_config
%run ../00_setup/00_utils
```

### Functions provided
- `create_schemas()` — create all schemas under rcm_platform catalog
- `log_pipeline_run()` — write audit row after every notebook completes
- `fetch_cms_paginated()` — paginated CMS API fetcher with retry logic
- `run_dq_checks()` — data quality checker, returns pass/fail + quarantine
- `write_delta()` — standardised Delta writer with logging
- `optimize_table()` — OPTIMIZE + ZORDER + VACUUM wrapper
- `get_row_count()` — safe row counter with logging

| Field       | Value |
|-------------|-------|
| Author      | Mayank Joshi |
| Layer       | Setup |
| Domain      | Healthcare — Revenue Cycle Management |
| Depends on  | 00_config |

In [0]:
%run /Workspace/Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_config

In [0]:
# ============================================================
# IMPORTS
# All libraries used across the project imported once here
# ============================================================

import requests
import time
import uuid
from datetime import datetime, date
from datetime import timezone

from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, lit, current_timestamp, current_date,
    sha2, concat_ws, when, coalesce, round,
    avg, sum, count, countDistinct, max, min,
    to_date, datediff, year, month, dayofmonth,
    date_format, regexp_replace, trim, upper,
    isnan, isnull
)
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType,
    BooleanType, TimestampType, DateType
)
from delta.tables import DeltaTable

print("All imports loaded successfully")

In [0]:
# ============================================================
# CREATE ALL SCHEMAS UNDER rcm_platform CATALOG
# Safe to re-run — uses IF NOT EXISTS throughout
# ============================================================

def create_schemas():
    """
    Creates all schemas (databases) under the rcm_platform catalog.
    Safe to re-run at any time — idempotent.
    Called once from 00_setup_databases notebook.
    """
    schemas = [
        (SCHEMA_BRONZE, "Raw ingestion layer — append-only, full audit trail"),
        (SCHEMA_SILVER, "Cleansed, enriched and conformed layer with SCD dimensions"),
        (SCHEMA_GOLD,   "Analytics-ready star schema — KPIs, fact and dimension tables"),
        (SCHEMA_ML,     "ML feature store, experiment metadata and model outputs"),
        (SCHEMA_AUDIT,  "Pipeline run logs, DQ results and observability tables"),
    ]

    print(f"Creating schemas in catalog: {CATALOG}\n")
    for schema_name, comment in schemas:
        spark.sql(f"""
            CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema_name}
            COMMENT '{comment}'
        """)
        print(f"  OK  {CATALOG}.{schema_name}")

    print(f"\nAll schemas ready under {CATALOG}")

In [0]:
# ============================================================
# PIPELINE RUN LOGGER
# Every notebook calls this at the end to record its execution
# ============================================================

def log_pipeline_run(
    notebook_name: str,
    layer: str,
    status: str,            # SUCCESS | FAILED | WARNING
    rows_read: int    = 0,
    rows_written: int = 0,
    rows_quarantined: int = 0,
    message: str      = ""
):
    """
    Writes one row to rcm_audit.pipeline_run_log after each notebook run.
    Gives full observability — you can see exactly what ran, when,
    how many rows moved, and whether anything failed.

    Args:
        notebook_name   : name of the calling notebook
        layer           : bronze | silver | gold | ml | setup
        status          : SUCCESS | FAILED | WARNING
        rows_read       : rows read from source
        rows_written    : rows written to target table
        rows_quarantined: rows sent to DQ quarantine
        message         : any additional context or error message
    """
    run_id = str(uuid.uuid4())
    logged_at = datetime.now(timezone.utc).replace(tzinfo=None)

    log_schema = StructType([
        StructField("run_id",           StringType(),    False),
        StructField("pipeline_name",    StringType(),    False),
        StructField("pipeline_version", StringType(),    False),
        StructField("environment",      StringType(),    False),
        StructField("notebook_name",    StringType(),    False),
        StructField("layer",            StringType(),    False),
        StructField("status",           StringType(),    False),
        StructField("rows_read",        IntegerType(),   True),
        StructField("rows_written",     IntegerType(),   True),
        StructField("rows_quarantined", IntegerType(),   True),
        StructField("message",          StringType(),    True),
        StructField("logged_at",        TimestampType(), False),
    ])

    log_data = [(
        run_id,
        PIPELINE_NAME,
        PIPELINE_VERSION,
        PIPELINE_ENV,
        notebook_name,
        layer,
        status,
        rows_read,
        rows_written,
        rows_quarantined,
        message,
        logged_at
    )]

    df_log = spark.createDataFrame(log_data, schema=log_schema)

    df_log.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(TBL_AUDIT_PIPELINE_LOG)

    print(f"\n[AUDIT LOG]")
    print(f"  Notebook  : {notebook_name}")
    print(f"  Status    : {status}")
    print(f"  Rows in   : {rows_read:,}")
    print(f"  Rows out  : {rows_written:,}")
    print(f"  Quarantine: {rows_quarantined:,}")
    if message:
        print(f"  Message   : {message}")
    print(f"  Run ID    : {run_id}")

In [0]:
# ============================================================
# CMS API PAGINATED FETCHER
# Handles pagination, retries and timeout automatically
# ============================================================

def fetch_cms_paginated(
    url: str,
    limit: int  = API_PAGE_LIMIT,
    timeout: int = API_TIMEOUT_SECS,
    max_retries: int = 3
) -> list:
    """
    Fetches all pages from a CMS Open Data API endpoint.
    Handles pagination automatically — keeps fetching until
    the API returns fewer rows than the page limit.

    Args:
        url         : CMS API endpoint URL
        limit       : rows per page (max 1000 for CMS)
        timeout     : seconds before request times out
        max_retries : number of retry attempts per page on failure

    Returns:
        list of dicts — all records across all pages
    """
    all_records = []
    offset      = 0
    page        = 1

    print(f"Fetching from CMS API...")
    print(f"  URL     : {url[:70]}...")
    print(f"  Page sz : {limit} rows\n")

    while True:
        params  = {"limit": limit, "offset": offset}
        attempt = 0

        while attempt < max_retries:
            try:
                response = requests.get(url, params=params, timeout=timeout)
                response.raise_for_status()
                data    = response.json()
                records = data.get("results", data.get("data", []))
                break
            except requests.exceptions.RequestException as e:
                attempt += 1
                if attempt == max_retries:
                    raise RuntimeError(
                        f"CMS API failed after {max_retries} attempts "
                        f"on page {page}. Error: {e}"
                    )
                wait = 2 ** attempt
                print(f"  Retry {attempt}/{max_retries} — waiting {wait}s...")
                time.sleep(wait)

        if not records:
            print(f"  No more records at offset {offset}. Done.")
            break

        all_records.extend(records)
        print(f"  Page {page:>3} — fetched {len(records):>5} rows "
              f"| total so far: {len(all_records):>7,}")

        if len(records) < limit:
            print(f"  Last page reached.")
            break

        offset += limit
        page   += 1

    print(f"\nTotal records fetched: {len(all_records):,}")
    return all_records

In [0]:
# ============================================================
# DATA QUALITY CHECKER
# Runs all DQ rules, returns clean df + writes failures to quarantine
# ============================================================

def run_dq_checks(
    df: DataFrame,
    source_table: str,
    notebook_name: str
) -> DataFrame:
    """
    Runs data quality checks on a DataFrame.
    Clean rows are returned for further processing.
    Failed rows are written to the quarantine table with a failure reason.

    Checks performed:
      1. Not-null checks on required columns
      2. Charge amount within valid range
      3. Payment amount positive
      4. Charge-to-payment ratio within bounds
      5. Discharge count positive

    Args:
        df            : input DataFrame to validate
        source_table  : name of source table (for quarantine metadata)
        notebook_name : calling notebook (for audit log)

    Returns:
        DataFrame — only rows that passed all DQ checks
    """
    total_rows = df.count()
    print(f"\n[DQ CHECKS] Running on {total_rows:,} rows from {source_table}")

    # ── Build a failure reason column ──
    # Each check appends to the reason string if it fails
    df_checked = df.withColumn("_dq_fail_reason", lit(""))

    # 1. Not-null checks
    for col_name in DQ_NOT_NULL_COLS:
        if col_name in df.columns:
            df_checked = df_checked.withColumn(
                "_dq_fail_reason",
                when(
                    col(col_name).isNull() | (trim(col(col_name).cast("string")) == ""),
                    concat_ws(" | ", col("_dq_fail_reason"),
                              lit(f"NULL:{col_name}"))
                ).otherwise(col("_dq_fail_reason"))
            )

    # 2. Charge amount range
    if "avg_submitted_charge" in df.columns:
        df_checked = df_checked.withColumn(
            "_dq_fail_reason",
            when(
                col("avg_submitted_charge").cast("double") < DQ_MIN_CHARGE_AMOUNT,
                concat_ws(" | ", col("_dq_fail_reason"), lit("CHARGE_TOO_LOW"))
            ).when(
                col("avg_submitted_charge").cast("double") > DQ_MAX_CHARGE_AMOUNT,
                concat_ws(" | ", col("_dq_fail_reason"), lit("CHARGE_TOO_HIGH"))
            ).otherwise(col("_dq_fail_reason"))
        )

    # 3. Payment amount positive
    if "avg_total_payment" in df.columns:
        df_checked = df_checked.withColumn(
            "_dq_fail_reason",
            when(
                col("avg_total_payment").cast("double") < DQ_MIN_PAYMENT_AMOUNT,
                concat_ws(" | ", col("_dq_fail_reason"), lit("PAYMENT_TOO_LOW"))
            ).otherwise(col("_dq_fail_reason"))
        )

    # 4. CTP ratio
    if "avg_submitted_charge" in df.columns and "avg_total_payment" in df.columns:
        df_checked = df_checked.withColumn(
            "_ctp_check",
            col("avg_submitted_charge").cast("double") /
            when(col("avg_total_payment").cast("double") == 0, lit(1))
            .otherwise(col("avg_total_payment").cast("double"))
        ).withColumn(
            "_dq_fail_reason",
            when(
                col("_ctp_check") > DQ_MAX_CTP_RATIO,
                concat_ws(" | ", col("_dq_fail_reason"), lit("CTP_RATIO_EXCEEDED"))
            ).otherwise(col("_dq_fail_reason"))
        ).drop("_ctp_check")

    # ── Split passed vs failed ──
    df_passed = df_checked.filter(
        col("_dq_fail_reason").isNull() | (trim(col("_dq_fail_reason")) == "")
    ).drop("_dq_fail_reason")

    df_failed = df_checked.filter(
        col("_dq_fail_reason").isNotNull() &
        (trim(col("_dq_fail_reason")) != "")
    )

    passed_count     = df_passed.count()
    quarantine_count = df_failed.count()

    # ── Write failures to quarantine table ──
    if quarantine_count > 0:
        df_failed \
            .withColumn("_source_table",   lit(source_table)) \
            .withColumn("_quarantined_at", current_timestamp()) \
            .withColumn("_notebook",       lit(notebook_name)) \
            .write \
            .format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .saveAsTable(TBL_BRONZE_QUARANTINE)

    print(f"  Total rows     : {total_rows:,}")
    print(f"  Passed DQ      : {passed_count:,}")
    print(f"  Quarantined    : {quarantine_count:,}  "
          f"({'%.1f' % (quarantine_count/total_rows*100)}%)")

    return df_passed

In [0]:
# ============================================================
# STANDARDISED DELTA WRITER
# Consistent write pattern used across every notebook
# ============================================================

def write_delta(
    df: DataFrame,
    table_name: str,
    mode: str = "overwrite",
    partition_cols: list = None,
    zorder_cols: list = None,
    comment: str = ""
):
    """
    Writes a DataFrame to a Delta table with consistent settings.
    Runs OPTIMIZE + ZORDER automatically if zorder_cols provided.

    Args:
        df             : DataFrame to write
        table_name     : fully qualified table name (catalog.schema.table)
        mode           : overwrite | append | merge (default: overwrite)
        partition_cols : list of columns to partition by
        zorder_cols    : list of columns to ZORDER by after write
        comment        : table comment / description
    """
    row_count = df.count()
    print(f"\n[WRITE DELTA] {table_name}")
    print(f"  Mode     : {mode}")
    print(f"  Rows     : {row_count:,}")

    writer = df.write \
        .format("delta") \
        .mode(mode) \
        .option("overwriteSchema", "true") \
        .option("mergeSchema",     "true")

    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
        print(f"  Partitions: {partition_cols}")

    writer.saveAsTable(table_name)

    # Add table comment if provided
    if comment:
        spark.sql(f"""
            COMMENT ON TABLE {table_name}
            IS '{comment}'
        """)

    # Run OPTIMIZE + ZORDER if columns specified
    if zorder_cols:
        zcols = ", ".join(zorder_cols)
        print(f"  ZORDER by : {zorder_cols}")
        spark.sql(f"OPTIMIZE {table_name} ZORDER BY ({zcols})")

    print(f"  Done      : {table_name}")
    return row_count

In [0]:
# ============================================================
# TABLE UTILITIES
# Safe helpers for common operations
# ============================================================

def get_row_count(table_name: str) -> int:
    """Returns row count for a Delta table. Returns 0 if table doesn't exist."""
    try:
        count = spark.table(table_name).count()
        print(f"  {table_name}: {count:,} rows")
        return count
    except Exception:
        print(f"  {table_name}: table does not exist yet")
        return 0


def table_exists(table_name: str) -> bool:
    """Returns True if a Delta table exists."""
    try:
        spark.table(table_name).limit(1).collect()
        return True
    except Exception:
        return False


def optimize_table(table_name: str, zorder_cols: list = None):
    """
    Runs OPTIMIZE and optional ZORDER on a Delta table.
    Call this on Gold tables after every full refresh.
    """
    print(f"\n[OPTIMIZE] {table_name}")
    if zorder_cols:
        zcols = ", ".join(zorder_cols)
        spark.sql(f"OPTIMIZE {table_name} ZORDER BY ({zcols})")
        print(f"  ZORDER by : {zorder_cols}")
    else:
        spark.sql(f"OPTIMIZE {table_name}")
    print(f"  Done")


def vacuum_table(table_name: str, retain_hours: int = 168):
    """
    Runs VACUUM on a Delta table.
    Default retention is 7 days (168 hours) — DO NOT go below 168
    unless you have disabled the retention check (not recommended).
    """
    print(f"\n[VACUUM] {table_name} — retaining {retain_hours}h")
    spark.sql(f"VACUUM {table_name} RETAIN {retain_hours} HOURS")
    print(f"  Done")


def show_table_history(table_name: str, limit: int = 5):
    """Shows the last N Delta transaction log entries for a table."""
    print(f"\n[HISTORY] {table_name} — last {limit} operations")
    display(spark.sql(f"DESCRIBE HISTORY {table_name} LIMIT {limit}"))


print("Table utilities loaded")

In [0]:
# ============================================================
# VOLUME SETUP — Unity Catalog Volumes
# Replaces DBFS — this is the Databricks modern standard
# ============================================================

def create_dbfs_dirs():
    """
    Creates a Unity Catalog Volume and all required subdirectories.
    Volumes are the production replacement for DBFS in Databricks.
    Safe to re-run — uses IF NOT EXISTS.
    """
    # Create the volume under rcm_bronze schema
    spark.sql(f"""
        CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA_BRONZE}.{VOLUME_NAME}
        COMMENT 'Raw file storage for RCM pipeline — inpatient, outpatient and hospital CSVs'
    """)
    print(f"OK  Volume: {CATALOG}.{SCHEMA_BRONZE}.{VOLUME_NAME}")

    # Create subdirectories by writing and deleting a placeholder file
    import os
    dirs = [
        RAW_INPATIENT_PATH,
        RAW_OUTPATIENT_PATH,
        RAW_HOSPITAL_PATH,
        CHECKPOINT_PATH,
        SCHEMA_PATH,
        TEMP_PATH,
    ]

    print("\nCreating volume subdirectories...")
    for d in dirs:
        placeholder = f"{d}/.keep"
        dbutils.fs.put(placeholder, "placeholder", overwrite=True)
        print(f"  OK  {d}")

    print(f"\nAll volume directories ready under:")
    print(f"  {BASE_VOLUME_PATH}")

In [0]:
# ============================================================
# UTILS LOAD CONFIRMATION
# ============================================================

print("=" * 60)
print("  RCM INTELLIGENCE PLATFORM — UTILS LOADED")
print("  Functions available:")
print("    create_schemas()")
print("    log_pipeline_run()")
print("    fetch_cms_paginated()")
print("    run_dq_checks()")
print("    write_delta()")
print("    optimize_table()")
print("    vacuum_table()")
print("    get_row_count()")
print("    table_exists()")
print("    create_dbfs_dirs()")
print("    show_table_history()")
print("=" * 60)